<a href="https://colab.research.google.com/github/hongxu-yn/Drought-monitoring-and-assessment/blob/main/step00_data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# 1. 环境准备与依赖安装
# ==========================================
!pip install -qU tqdm geemap earthengine-api

import ee
import geemap
import os
from google.colab import drive
from tqdm.auto import tqdm


drive.mount('/content/drive')

try:
    ee.Initialize(project='ee-hongxu')
except:
    ee.Authenticate()
    ee.Initialize(project='ee-hongxu')

roi = ee.Geometry.Rectangle([97, 20, 107, 30])

def preprocess_ndvi(image):
    ndvi = image.select('NDVI')
    mask = ndvi.neq(-3000)
    ndvi_real = ndvi.updateMask(mask).multiply(0.0001)
    return ndvi_real.copyProperties(image, ['system:time_start', 'system:index'])

collection = ee.ImageCollection('MODIS/061/MOD13A3') \
    .filterBounds(roi) \
    .filterDate('2023-01-01', '2023-12-31') \
    .map(preprocess_ndvi)

out_dir = '/content/drive/MyDrive/DATA/MOD13A3'
os.makedirs(out_dir, exist_ok=True)

image_list = collection.toList(collection.size())
count = collection.size().getInfo()
print(f"\n🚀 准备下载 {count} 幅处理后的影像到: {out_dir}")

# 带进度条的循环下载
for i in tqdm(range(count), desc="下载进度", unit="月"):
    image = ee.Image(image_list.get(i))

    date = ee.Date(image.get('system:time_start')).format('YYYY_MM').getInfo()
    file_name = f'MODIS_NDVI_{date}.tif'
    out_path = os.path.join(out_dir, file_name)

    if os.path.exists(out_path):
        continue

    geemap.ee_export_image(
        image,
        filename=out_path,
        scale=1000,
        region=roi,
        file_per_band=False
    )

print("\n✨ 所有影像下载完成！(无效值已被掩膜，有效值范围为 -0.2 到 1.0)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.9/481.9 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 81.4 MB/s eta 0:00:00
Mounted at /content/drive
